# SPHEREx self-calibration: one channel, end to end

A run is a TOML config: an **instrument** (here `spherex`: its LVF subchannel/column chunk
maps and its band-centre / band-width maps), a **mode** (the model the solver fits) and a
**task** (`reproject`, `cal`, `mosaic`, `npass`). This notebook reprojects (optional), calibrates
one channel of Detector 3, reads the calibration with `CalFile`, and looks at the mosaic; then it
shows the model the mode built and a spectral model on Detector 4.

The paths below point at the Detector 3 NEP frames already reprojected on this machine (300 of
them are used, as in the regression gate). Change them, or set the `SELFCAL_DEMO_*` environment
variables, for your own run.

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

REPO = os.path.abspath('..')                       # this notebook lives in <repo>/notebooks/
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from selfcal import set_progress
from selfcal.geometry.map_helper import chunk_to_det
from selfcal.io.calfile import CalFile
from selfcal_scripts.runner import pipelines
from selfcal_scripts.runner.config import load_config
from selfcal_scripts.runner.engine import RunContext

plt.rcParams['figure.dpi'] = 300
set_progress(False)


def run_config(toml_text, name):
    """Write a run config (exactly what you would keep in a .toml file) and run it."""
    os.makedirs(CONFIG_DIR, exist_ok=True)
    path = os.path.join(CONFIG_DIR, name)
    with open(path, 'w') as f:
        f.write(toml_text)
    return pipelines.run(load_config(path)), path


def show(img, title, ax=None, lo=1, hi=99, cmap='viridis'):
    """An image with its colorbar, clipped to percentiles of its finite values."""
    ax = ax or plt.gca()
    good = img[np.isfinite(img) & (img != 0)]
    vmin, vmax = np.percentile(good, [lo, hi]) if good.size else (None, None)
    im = ax.imshow(img, origin='lower', vmin=vmin, vmax=vmax, cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=8)
    return im

OUTPUT_DIR = os.environ.get('SELFCAL_DEMO_OUTPUT', '/mnt/md124/thomasli/selfcal/outputs/')
RUN_NAME = 'SPHEREx_nep_qr2_det3_6p2arcsec'           # <OUTPUT_DIR>/<RUN_NAME>/ref.fits = the sky grid
FRAMES = os.environ.get('SELFCAL_DEMO_FRAMES',
                        '/home/thomasli/selfcal-project/selfcal-memopt/cache/reproj_nvme_SPHEREx_nep_qr2_det3_6p2arcsec')
CACHE_DIR = os.environ.get('SELFCAL_DEMO_CACHE', '/tmp/selfcal_demo_cache/')
CONFIG_DIR = os.path.join(CACHE_DIR, 'configs')
N_FRAMES = int(os.environ.get('SELFCAL_DEMO_N_FRAMES', 300))
WORKERS = int(os.environ.get('SELFCAL_DEMO_WORKERS', 16))

## 1. Reprojection (once per detector)

Every exposure is resampled onto one reference grid and stored as a frame file. The frames above already exist, so this cell only runs when `REPROJECT = True`.

In [ ]:
REPROJECT = False

reproject_toml = f"""
task = "reproject"
output_dir = "{OUTPUT_DIR}"
run_name = "{RUN_NAME}"
resolution_arcsec = 6.2
cache_dir = "{CACHE_DIR}"

[instrument]
name = "spherex"
detector = 3
num_sub = 10
num_ch = 34
num_col = 3

[reproject]
input_dirs = ["/mnt/md124/SPHEREx/SPHEREx_nep_data/qr2"]
file_pattern = "/*/*/*/*D3*.fits"
padding_pixels = 100
max_workers = 50
reproj_func = "exact"
padding_percentage = 0.05
"""
if REPROJECT:
    run_config(reproject_toml, 'reproject.toml')

## 2. Calibrate one channel

`mode = "continuum"`: a constant sky per pixel, a free offset per frame and chunk (smooth along the columns, a soft linear polynomial along them, the per-frame mean anchored at zero) and a per-frame scalar. The `cal` task solves and then builds the mosaic.

In [ ]:
cal_toml = f"""
task = "cal"
mode = "continuum"
output_dir = "{OUTPUT_DIR}"
run_name = "{RUN_NAME}"
resolution_arcsec = 6.2
cache_dir = "{CACHE_DIR}"
suffix = "_notebook_demo"
oversample = 2
reproj_override = "{FRAMES}"
n_frames = {N_FRAMES}
apply_n_threads = {WORKERS}

[instrument]
name = "spherex"
detector = 3
num_sub = 10
num_ch = 34
num_col = 3
channels = [[17]]

[params]
reg_weight = 0.1
poly_weight = 0.5
poly_degree = 1

[calibration]
apply_mask = true
apply_weight = false
outlier_thresh = 5.0
ignore_list = []
batch_size = 50
offset_regularization = true
weighted_damping = true
damp_weight = 0.1
max_workers = {WORKERS}

[lsqr]
atol = 1e-6
btol = 1e-6
damp = 0
iter_lim = 50
precondition = true
solver = "lsqr"

[mosaic]
apply_mask = true
apply_weight = false
make_std_map = true
apply_sigma_clipping = true
sigma = 2.0
ignore_list = [21]
cache_batch_size = 50
coadd_batch_size = 50
cache_intermediate = true
max_workers = {WORKERS}
"""
result, cal_config = run_config(cal_toml, 'cal.toml')
print(result.cal_paths, result.mosaic_paths)

## 3. What the solve produced

`CalFile` reads every calibration product (all schema versions): the sky maps, the per-frame offsets of each chunk map, the per-frame scalars, the frame list.

In [ ]:
with CalFile(result.cal_paths[0]) as cal:
    print(cal.describe())
    sky = cal.sky('continuum')
    offsets = cal.offsets[0]                 # (frames, chunks)
    scalar = cal.frame_scalar
    chunk_map = cal.chunk_maps[0]            # detector grid -> chunk id

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
show(sky, 'sky: continuum (MJy/sr)', ax=axes[0])
show(chunk_to_det(chunk_map, offsets[0]), 'offset of frame 0 on the detector (MJy/sr)', ax=axes[1],
     cmap='RdBu_r')
axes[2].plot(scalar, '.', ms=2)
axes[2].set_xlabel('frame')
axes[2].set_ylabel('per-frame scalar (MJy/sr)')
plt.tight_layout()

## 4. The mosaic

In [ ]:
with fits.open(result.mosaic_paths[0]) as hdul:
    print([h.name for h in hdul])
    mean = hdul['MEAN_MAP'].data
    clipped = hdul['SC_MEAN_MAP'].data

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
show(mean, 'mean of the calibrated frames (MJy/sr)', ax=axes[0])
show(clipped, 'sigma-clipped mean (MJy/sr)', ax=axes[1])
plt.tight_layout()

## 5. The recipe is a model

A mode is a producer of a `ModelSpec`: sky terms, offset terms and their priors. This is the model the `continuum` mode built for the run above; with `mode = "model"` the same thing is written directly in a `[model]` table.

In [ ]:
ctx = RunContext.build(load_config(cal_config))
spec = ctx.mode.spec(ctx.cfg, ctx.inst, ctx.geom)
for term in spec.sky:
    print('sky   ', term)
for term in spec.offset:
    print('offset', term)
print('scalar', spec.scalar)

## 6. A spectral model on Detector 4

Each extra sky term is a map times a coefficient of a data variable: here the aromatic PAH feature at 3.29 µm as a tabulated template of the wavelength (the instrument's band-centre map, alias `wavelength`) and a hard polynomial-basis offset along the subchannels, the production multi-line recipe with one line. Building the model against the Detector 4 geometry checks every name and shows what the solver would fit; running it needs the Detector 4 frames.

In [ ]:
from selfcal.models.spec import ModelSpec

TEMPLATES = os.path.join(REPO, 'selfcal', 'instruments', 'spherex', 'data', 'line_templates')
model = ModelSpec.from_config({
    'sky': [{'name': 'continuum'},
            {'name': 'aromatic', 'damp_weight': 5e-3,
             'coefficient': {'variable': 'wavelength', 'function': 'template',
                             'file': os.path.join(TEMPLATES, 'aromatic_3p289.npz')}}],
    'offset': [{'kind': 'polybasis', 'degree': 2, 'lo': 200, 'hi': 320}],
})
d4_instrument = dict(load_config(cal_config).instrument_cfg, detector=4)
geom = ctx.inst.detector_geometry(d4_instrument, 1)
model.check(geom)
sky_model = model.build_sky_model(geom)
print([(c.name, c.coefficient.describe() if c.coefficient else '1') for c in sky_model.components])
lam = np.linspace(3.1, 3.6, 400)
g = sky_model.components[1].coefficients({'BC': lam})
plt.figure(figsize=(5, 3))
plt.plot(lam, g)
plt.xlabel('wavelength (µm)')
plt.ylabel('coefficient of the aromatic map')
plt.tight_layout()